# 第3章　医療AIの主要ライブラリ ― 道具箱をそろえる

**『医療診断支援AIを自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 3.2　画像の入出力 ― Pillow と OpenCV

In [ ]:
from PIL import Image
import numpy as np
img = Image.open("fundus.jpg").convert("RGB")   # 画像を開く
arr = np.array(img)                              # NumPy配列へ (H, W, 3)

import cv2
bgr = cv2.imread("xray.png")                     # OpenCVはBGR順で読む点に注意
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)     # グレースケール化
clahe = cv2.createCLAHE(clipLimit=2.0)           # 局所コントラスト強調(眼底で有効)
enhanced = clahe.apply(gray)

## 3.3　医療画像フォーマット ― pydicom・nibabel・SimpleITK

In [ ]:
import pydicom
ds = pydicom.dcmread("ct_slice.dcm")
pixel = ds.pixel_array                                  # 画素の配列
hu = pixel * ds.RescaleSlope + ds.RescaleIntercept      # CT値(HU)へ変換
print(ds.PatientID, ds.Modality, ds.Rows, ds.Columns)   # メタデータのタグ

In [ ]:
import nibabel as nib
vol = nib.load("liver.nii.gz")
data = vol.get_fdata()          # 3次元配列 (X, Y, Z)
# pydicomで積んだ (Z, H, W) とは軸順が逆。アキシャルは通常 data[:, :, k]
spacing = vol.header.get_zooms()  # 1ボクセルが何mmか(第7章のNIfTIと3次元データの節)

## 手を動かす ― CTの1症例を読み込み、3断面で表示する（ミニプロジェクト）

In [ ]:

# Colabなら先頭で: !pip install pydicom
# ドライブに置いた症例を使うなら: from google.colab import drive; drive.mount('/content/drive')
import pydicom, numpy as np, glob, os

def load_ct_series(folder):   # 例: /content/drive/MyDrive/ct_case01 に .dcm を並べておく
    """1症例分のDICOMスライス群を、正しい順序で立体に積む"""
    slices = [pydicom.dcmread(p) for p in glob.glob(os.path.join(folder, "*.dcm"))]
    # ① 罠その1：位置情報で並べ替える（ファイル名順ではない）。ただし下の [2]（体軸座標）で
    #    並べてよいのはアキシャル収集のみ。冠状断・矢状断の再構成では全スライスで同値になり、
    #    例外も警告も出ないまま並べ替えが効かない（入門編の自分の施設の画像で試す章）。
    # ここで扱うのは「単一シリーズ・単フレーム・アキシャル収集のCT」だけ。
    # 対応外の入力は、黙って通さず例外で止める（assert は -O で無効化されるので使わない）。
    def reject(cond, msg):
        if cond:
            raise ValueError("この読込関数の対応外です: " + msg)
    reject(len(slices) == 0, "DICOMが見つからない")
    reject(len({s.SeriesInstanceUID for s in slices}) != 1, "複数シリーズが混在")
    reject(any(getattr(s, "Modality", "") != "CT" for s in slices), "CT以外のモダリティ")
    reject(any(int(getattr(s, "NumberOfFrames", 1)) != 1 for s in slices), "マルチフレーム")
    reject(len({(s.Rows, s.Columns) for s in slices}) != 1, "画像サイズが揃っていない")
    # 方向余弦がアキシャル（行=x軸、列=y軸）でなければ、z座標で並べる前提が崩れる
    for s in slices:
        iop = [round(float(v), 3) for v in s.ImageOrientationPatient]
        reject(iop[:3] != [1.0, 0.0, 0.0] or iop[3:] != [0.0, 1.0, 0.0],
               f"アキシャル収集でない（ImageOrientationPatient={iop}）")
        reject("RescaleSlope" not in s or "RescaleIntercept" not in s, "HU変換情報が無い")
        reject(any((not np.isfinite(float(v))) or float(v) <= 0 for v in s.PixelSpacing), "PixelSpacing が不正")
    # 全スライスが同じ直交格子に載っていること：面内の画素間隔と、面内の原点（x, y）が一致する
    ps0 = tuple(round(float(v), 4) for v in slices[0].PixelSpacing)
    xy0 = tuple(round(float(v), 2) for v in slices[0].ImagePositionPatient[:2])
    reject(any(tuple(round(float(v), 4) for v in s.PixelSpacing) != ps0 for s in slices), "PixelSpacing がスライス間で異なる")
    reject(any(tuple(round(float(v), 2) for v in s.ImagePositionPatient[:2]) != xy0 for s in slices), "面内の原点がスライス間でずれている")
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))   # アキシャルなので z で並べてよい
    # 変換係数はスライスごとに違いうる。先頭の値を全枚に掛けてはいけない。
    # int16 への先行変換もしない（元の値域を保証できない）。
    hu = np.stack([
        s.pixel_array.astype(np.float32) * float(s.RescaleSlope) + float(s.RescaleIntercept)
        for s in slices])                                              # (Z, H, W)
    # ② 罠その2：1ボクセルの実寸(mm)。面内とスライス間隔は普通は違う
    # スライス「間隔」は、隣り合うスライスの体軸方向の位置の差から求める。
    # SliceThickness は“厚み”で、間隔とは別物（重なり再構成や隙間があるとズレる）。
    zs = [float(s.ImagePositionPatient[2]) for s in slices]
    if len(zs) > 1:
        gaps = np.diff(zs)
        # 先頭2枚の差だけで決めない。等間隔でなければ、そのまま積んではいけない。
        reject(len(set(zs)) != len(zs), "同じ位置のスライスが重複している")
        reject(not np.allclose(gaps, gaps[0], atol=1e-3), f"スライス間隔が不均一: {gaps[:5]}")
        dz = abs(float(gaps[0]))
    else:
        raise ValueError("この3D読込例には2枚以上のスライスが必要です")
    dy, dx = map(float, slices[0].PixelSpacing)
    return hu.astype(np.float32), (dz, dy, dx)
# 対応外の入力（CT以外、マルチフレーム、斜位収集、シリーズ混在、非等間隔、面内の間隔や原点の不一致、HU変換情報の欠落、スライスが1枚だけ）は、
# 黙って通さず、上の reject で例外にして止める。斜位を扱うなら、位置を断面の法線へ射影して
# 並べる別の実装が要る（本章では扱わない）。

hu, (dz, dy, dx) = load_ct_series("/content/drive/MyDrive/ct_case01")
print("ボリュームの形:", hu.shape, " HU範囲:", int(hu.min()), "〜", int(hu.max()))
print(f"1ボクセル = {dz}mm(スライス間) × {dy}mm × {dx}mm(面内)")
# ボリュームの形: (64, 512, 512)  HU範囲: -1024 〜 1918
# 1ボクセル = 5.0mm(スライス間) × 0.7mm × 0.7mm(面内)

In [ ]:
import matplotlib.pyplot as plt

def window(hu, center, width):
    lo, hi = center - width/2, center + width/2
    return np.clip((hu - lo) / (hi - lo), 0, 1)

soft = window(hu, center=40, width=400)          # 腹部軟部ウィンドウ
z, y, x = [s // 2 for s in hu.shape]             # 各軸の中央スライス

fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(soft[z], cmap="gray")                              # アキシャル（体の断面）
ax[1].imshow(soft[:, y], cmap="gray", aspect=dz / dx)          # コロナル（列はX方向なのでdx）
ax[2].imshow(soft[:, :, x], cmap="gray", aspect=dz / dy)       # サジタル（列はY方向なのでdy）
for a, t in zip(ax, ["axial", "coronal", "sagittal"]):   # 図中は英語（日本語は□に化ける）
    a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 3.4　機械学習と評価 ― scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score

# 陽性・陰性の比率を保ったまま分割（層化）― 不均衡データで重要(第5章)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

auc = roc_auc_score(y_true, y_score)     # ROC-AUC
f1  = f1_score(y_true, y_pred)           # F1スコア
cm  = confusion_matrix(y_true, y_pred)   # 混同行列(社会実装編の評価の章)

## 3.5　データ拡張 ― albumentations

In [ ]:
import albumentations as A
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])                                        # mask= に渡すだけで、画像とマスクが同じ変形で動く

out = transform(image=img, mask=mask)
aug_img, aug_mask = out["image"], out["mask"]

## 3.6　ディープラーニングのフレームワーク ― PyTorch と TensorFlow

In [ ]:
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([                       # 層を積むだけ
    keras.Input(shape=(224, 224, 3)),          # Keras 3 系では入力を Input で先に宣言する
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(2, activation="softmax"),  # 2クラス分類
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=5)                     # 学習も1行

## 名前の付け方に、規則がある

In [ ]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged
from monai.losses import DiceLoss

# CTを読み込み、腹部ウィンドウで正規化する前処理をまとめて定義
prep = Compose([
    LoadImaged(keys=["image", "label"]),            # DICOM/NIfTIを読む（両方に適用）
    EnsureChannelFirstd(keys=["image", "label"]),   # 先頭にチャネル軸を足す（空間軸の順は読み込んだファイルのまま）
    ScaleIntensityRanged(keys=["image"],            # ← 画像だけ。ラベルには絶対にかけない
                         a_min=-160, a_max=240,     # 腹部ウィンドウ（HU値）
                         b_min=0.0, b_max=1.0, clip=True),
])
loss_fn = DiceLoss(to_onehot_y=True, softmax=True)  # セグメンテーション用のDice損失（第11章）

## 3.8　学習を見守る道具 ― tqdm と実験管理

In [ ]:
from tqdm import tqdm
for images, labels in tqdm(train_loader):   # ループに巻くだけで進捗バーが出る
    ...